[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C31_Coding_Agent_Course/05_coding_agent/05_coding_agent.ipynb)

# 05 · 完整编码 Agent（集大成）

目标：把前四模块的工具组装成一个**完整编码 agent**——**工具注册表 + agent 主循环（tool_use→执行→回填 tool_result）+ 系统提示**——用 **MockLLM** 在 `tempfile` 玩具仓里**端到端真跑 pytest 修一个 bug**；做 **SWE-bench 式评测**；附**真实 Claude（Messages API）适配**（无 key 自动回退）。

路线：工具注册表 → agent 主循环 → 系统提示 → 端到端修 bug(真跑) → SWE-bench 式评测 → 真实 Claude 适配→ ✏️ 练习 → 📖 答案 → 🧪 真实 SWE 式胶囊。

> 心智模型：**agent = 一组工具 + 一张注册表 + 一个循环 + 一套纪律 + 一个客观成功判据。不到 200 行，不依赖框架，真能修 bug。**

## 0 · 准备：把前四模块的工具集中起来

组装 agent 前，先把模块 01（文件）、02（shell/pytest）、03（搜索）的精简版工具都备好——它们是 agent 的「能力库」。

In [ ]:
import os, sys, re, tempfile, shutil, subprocess

# ---- 模块 01：文件工具 ----
def read_file(work, path):
    lines = open(os.path.join(work, path), encoding='utf-8').read().splitlines()
    return '\n'.join(f'{i+1:4d}| {ln}' for i, ln in enumerate(lines))
def edit_file(work, path, old, new):
    full = os.path.join(work, path); text = open(full, encoding='utf-8').read()
    cnt = text.count(old)
    if cnt != 1:
        raise ValueError(f'旧串匹配 {cnt} 处，需恰好 1 处（请加上下文使唯一）')
    open(full, 'w', encoding='utf-8').write(text.replace(old, new))
    return f'已编辑 {path}：替换 1 处'

# ---- 模块 02：shell 跑 pytest（--color=no 让输出干净可解析）----
def run_tests(work, timeout=60):
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no'],
                       cwd=work, capture_output=True, text=True, timeout=timeout,
                       env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
    out = (r.stdout + r.stderr)
    tail = out.strip().splitlines()[-1] if out.strip() else ''
    return f"returncode={r.returncode}\n{tail}"
def tests_pass(work):
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no'],
                       cwd=work, capture_output=True, text=True, timeout=60,
                       env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
    return r.returncode == 0

# ---- 模块 03：代码搜索（精简）----
def code_search(work, query):
    hits = []
    for root, dirs, files in os.walk(work):
        dirs[:] = [d for d in dirs if d not in {'.git','__pycache__'} and not d.startswith('.')]
        for fn in files:
            if not fn.endswith('.py'): continue
            rel = os.path.relpath(os.path.join(root, fn), work)
            for i, line in enumerate(open(os.path.join(root, fn), encoding='utf-8', errors='replace'), 1):
                if re.search(re.escape(query), line):
                    hits.append(f'{rel}:{i}: {line.strip()}')
    return '\n'.join(hits) if hits else f'未找到 {query!r}'

print('能力库就绪：read_file / edit_file / run_tests / code_search ✅')

## 1 · 工具注册表：把能力登记成一张表 ⭐

登记每个工具的 **名字 + 实现 + schema**。两个关键：**分发时捕获一切异常**(不让工具崩掉循环)、能吐出 **schema 清单**(喂给 LLM)。

In [ ]:
class ToolRegistry:
    def __init__(self, work):
        self.work = work
        self.tools = {}
    def register(self, name, fn, schema):
        self.tools[name] = {'fn': fn, 'schema': schema}
    def call(self, name, args):
        if name not in self.tools:
            return f'错误：未知工具 {name}。可用：{list(self.tools)}'
        try:
            return str(self.tools[name]['fn'](self.work, **args))
        except Exception as e:
            return f'工具 {name} 执行出错：{type(e).__name__}: {e}'
    def schemas(self):
        return [t['schema'] for t in self.tools.values()]

def make_schema(name, desc, props, required):
    return {'name': name, 'description': desc,
            'input_schema': {'type': 'object', 'properties': props, 'required': required}}

WORK = tempfile.mkdtemp(prefix='c31_agent_')
reg = ToolRegistry(WORK)
reg.register('read_file', read_file,
    make_schema('read_file', '读取文件内容(带行号)',
                {'path': {'type':'string'}}, ['path']))
reg.register('edit_file', edit_file,
    make_schema('edit_file', '把文件中唯一匹配的 old 替换为 new(old 须唯一)',
                {'path':{'type':'string'},'old':{'type':'string'},'new':{'type':'string'}},
                ['path','old','new']))
reg.register('run_tests', run_tests,
    make_schema('run_tests', '运行 pytest，返回返回码与摘要', {}, []))
reg.register('code_search', code_search,
    make_schema('code_search', '在代码库搜索字符串，返回命中位置',
                {'query':{'type':'string'}}, ['query']))

print('注册的工具:', list(reg.tools))
assert len(reg.schemas()) == 4
# 分发：未知工具 / 出错都返回友好字符串而非崩溃
assert '未知工具' in reg.call('nope', {})
assert '执行出错' in reg.call('read_file', {'path': 'no_such_file.py'})
print('✅ 注册表就绪：按名字分发、捕获异常转友好错误、能吐 schema 清单')

## 2 · agent 主循环：tool_use → 执行 → 回填 tool_result ⭐⭐

模块 00 骨架的完整版。LLM 要么请求工具(继续)、要么给最终答复(结束)；消息回填**贴合 Anthropic tool_use/tool_result 格式**，这样换真实 Claude 时循环不用改。

In [ ]:
class MockLLM:
    '''确定性替身：按脚本依次给出动作。
       动作: {'type':'tool','name':..,'args':..,'id':..} 或 {'type':'final','text':..}'''
    def __init__(self, script):
        self.script = list(script); self.i = 0
    def step(self, messages, tool_schemas):
        if self.i >= len(self.script):
            return {'type': 'final', 'text': '(脚本结束)'}
        a = self.script[self.i]; self.i += 1
        return a

def run_agent(llm, registry, task, max_steps=15, verbose=True):
    messages = [{'role': 'user', 'content': task}]
    trajectory = []
    for step in range(max_steps):
        action = llm.step(messages, registry.schemas())
        if action['type'] == 'final':
            return {'done': True, 'steps': step, 'answer': action['text'], 'trajectory': trajectory}
        name, args, tid = action['name'], action['args'], action.get('id', f'call_{step}')
        result = registry.call(name, args)
        trajectory.append((name, args, result))
        if verbose:
            print(f'  [步骤{step}] {name}({args}) -> {result.splitlines()[0][:60]}')
        messages.append({'role':'assistant','content':[
            {'type':'tool_use','id':tid,'name':name,'input':args}]})
        messages.append({'role':'user','content':[
            {'type':'tool_result','tool_use_id':tid,'content':result}]})
    return {'done': False, 'steps': max_steps, 'answer': '(到达步数上限)', 'trajectory': trajectory}

# 烟雾测试：一个只 read 再 final 的最小脚本
open(os.path.join(WORK, 'demo.py'), 'w').write('x = 1\n')
llm = MockLLM([{'type':'tool','name':'read_file','args':{'path':'demo.py'},'id':'t1'},
               {'type':'final','text':'看完了'}])
res = run_agent(llm, reg, '读一下 demo.py', verbose=False)
assert res['done'] and res['steps'] == 1
assert res['trajectory'][0][0] == 'read_file' and 'x = 1' in res['trajectory'][0][2]
print('✅ 主循环就绪：tool_use→执行→回填(贴合 Anthropic 格式)→final 结束')

## 3 · 系统提示：给 agent 定下行为纪律

把「先理解再改、改完必验、**不准改测试作弊**」等工程纪律写成自然语言注入 LLM。它是 agent 的「岗前手册」。

In [ ]:
SYSTEM_PROMPT = '''你是一个编码 agent，任务是修复给定代码库里的 bug，让测试通过。
工作流：
1. 先用 code_search / read_file 理解相关代码，再动手改。
2. 用 edit_file 做精确修改（old 字符串须在文件中唯一）。
3. 每次修改后用 run_tests 跑测试验证。
纪律：
- 只修复【实现代码】，绝不通过修改测试用例来让测试通过。
- 测试全部通过后，简要说明改动并结束。'''

print(SYSTEM_PROMPT)
assert '绝不通过修改测试' in SYSTEM_PROMPT, '必须含防作弊纪律'
assert 'run_tests' in SYSTEM_PROMPT and '理解' in SYSTEM_PROMPT
print('\n✅ 系统提示就绪：含工作流 + 防 reward hacking 红线（真实 Claude 会读它）')

## 4 · 端到端：让 agent 真的修好一个 bug ⭐⭐⭐

把注册表 + 主循环 + 系统提示拼起来，跑一个完整任务。MockLLM 扮演合理的决策序列，**每个工具调用都在玩具仓里真实执行**：真跑 pytest、真读、真改、真再跑——看着 bug 由红转绿。

In [ ]:
# 带 bug 的玩具仓库
open(os.path.join(WORK, 'calc.py'), 'w').write(
    'def add(a, b):\n    return a - b   # BUG: 应当是 a + b\n')
open(os.path.join(WORK, 'test_calc.py'), 'w').write(
    'from calc import add\n\ndef test_add():\n    assert add(2, 3) == 5\n')
assert tests_pass(WORK) is False, '初始有 bug，测试应红'

# MockLLM 扮演 agent 的合理决策序列：测→搜→读→改→再测→收尾
script = [
    {'type':'tool','name':'run_tests','args':{},'id':'s1'},                       # 看到红
    {'type':'tool','name':'code_search','args':{'query':'def add'},'id':'s2'},     # 定位
    {'type':'tool','name':'read_file','args':{'path':'calc.py'},'id':'s3'},        # 读实现
    {'type':'tool','name':'edit_file',                                            # 修复
     'args':{'path':'calc.py','old':'return a - b   # BUG: 应当是 a + b','new':'return a + b'},'id':'s4'},
    {'type':'tool','name':'run_tests','args':{},'id':'s5'},                       # 验证绿
    {'type':'final','text':'已修复 add 的符号错误(a-b → a+b)，测试通过。'},
]
llm = MockLLM(script)
print('=== agent 开始解决任务 ===')
result = run_agent(llm, reg, 'test_calc 失败了，请修复它。')
print('\n最终答复:', result['answer'])
# 验证：agent 真的把 bug 修好了
assert result['done'] is True
assert 'return a + b' in open(os.path.join(WORK, 'calc.py')).read()
assert tests_pass(WORK) is True, 'agent 修复后 pytest 应真的全绿'
print('\n✅✅✅ 端到端成功：完整 agent 真的把玩具仓的 bug 修好、pytest 真的变绿！')

## 5 · SWE-bench 式评测：补丁让测试通过 = 成功

客观评判 agent：跑一轮后**跑测试**——绿则 `resolved`。更严：**校验测试文件未被篡改**(防 reward hacking)、跑**全套**(防回归)。

In [ ]:
import hashlib
def test_files_fingerprint(work):
    '''给所有 test_*.py 内容算指纹，用于检测 agent 是否篡改了测试。'''
    h = hashlib.md5()
    for root, _, files in os.walk(work):
        for fn in sorted(files):
            if fn.startswith('test_') and fn.endswith('.py'):
                h.update(open(os.path.join(root, fn), 'rb').read())
    return h.hexdigest()

def evaluate(agent_fn, build_repo_fn):
    '''SWE-bench 式评测：建带 bug 的仓库→记录测试指纹→跑 agent→判定。
       resolved = 测试全绿 且 测试文件未被篡改。'''
    work = build_repo_fn()
    try:
        before_tests = test_files_fingerprint(work)
        agent_fn(work)                                  # 跑 agent（真改真测）
        after_tests = test_files_fingerprint(work)
        passed = tests_pass(work)
        tampered = (before_tests != after_tests)        # 测试被改 = 作弊
        resolved = passed and not tampered
        return {'resolved': resolved, 'tests_pass': passed, 'test_tampered': tampered}
    finally:
        shutil.rmtree(work, ignore_errors=True)

# 评测我们的 agent（用正确脚本，应判 resolved）
def build_repo():
    w = tempfile.mkdtemp(prefix='c31_eval_')
    open(os.path.join(w, 'calc.py'), 'w').write('def add(a, b):\n    return a - b\n')
    open(os.path.join(w, 'test_calc.py'), 'w').write(
        'from calc import add\n\ndef test_add():\n    assert add(2, 3) == 5\n')
    return w
def good_agent(work):
    r = ToolRegistry(work)
    r.register('edit_file', edit_file, make_schema('edit_file','',{},[]))
    r.register('run_tests', run_tests, make_schema('run_tests','',{},[]))
    llm = MockLLM([
        {'type':'tool','name':'edit_file',
         'args':{'path':'calc.py','old':'return a - b','new':'return a + b'},'id':'e1'},
        {'type':'final','text':'done'}])
    run_agent(llm, r, '修复 bug', verbose=False)

verdict = evaluate(good_agent, build_repo)
print('评测结果(正确 agent):', verdict)
assert verdict['resolved'] is True and verdict['test_tampered'] is False
print('✅ 评测就绪：补丁让测试真绿、且未篡改测试 → resolved')

**作弊检测演示**：一个偷懒 agent 改了测试让它「通过」——评测应判 `tests_pass=True` 但 `resolved=False`（因篡改了测试）。

In [ ]:
def cheating_agent(work):
    r = ToolRegistry(work)
    r.register('edit_file', edit_file, make_schema('edit_file','',{},[]))
    llm = MockLLM([
        # 不修实现，反而改测试断言来「通过」(作弊！)
        {'type':'tool','name':'edit_file',
         'args':{'path':'test_calc.py','old':'assert add(2, 3) == 5','new':'assert add(2, 3) == -1'},'id':'c1'},
        {'type':'final','text':'done(cheated)'}])
    run_agent(llm, r, '修复 bug', verbose=False)

cheat_verdict = evaluate(cheating_agent, build_repo)
print('评测结果(作弊 agent):', cheat_verdict)
assert cheat_verdict['tests_pass'] is True, '改了断言后测试确实「绿」了'
assert cheat_verdict['test_tampered'] is True, '但测试文件被篡改'
assert cheat_verdict['resolved'] is False, '篡改测试 → 不算解决（防 reward hacking）'
print('✅ 作弊被识破：改测试让它「通过」不算 resolved —— 评测必须不可被操纵')

---
## ✏️ 练习 1：给 agent 注册一个新工具（list_files）

实现 `list_files(work)`：返回工作区里所有 `.py` 文件的相对路径(每行一个)。然后把它注册进一个 `ToolRegistry`，并通过 `reg.call('list_files', {})` 验证它能被分发调用。

In [ ]:
def list_files(work):
    # TODO: os.walk 收集所有 .py 的相对路径，跳过 __pycache__/隐藏目录，'\n'.join 返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
w = tempfile.mkdtemp(prefix='c31_ex1_')
try:
    open(os.path.join(w, 'a.py'), 'w').write('x=1')
    os.makedirs(os.path.join(w, 'pkg'))
    open(os.path.join(w, 'pkg', 'b.py'), 'w').write('y=2')
    open(os.path.join(w, 'note.txt'), 'w').write('not py')
    r = ToolRegistry(w)
    r.register('list_files', list_files, make_schema('list_files','列出所有 .py',{},[]))
    out = r.call('list_files', {})
    print('list_files ->', out.split())
    files = set(out.split())
    assert 'a.py' in files and os.path.join('pkg','b.py') in files
    assert not any('note.txt' in f for f in files), '非 .py 不应列出'
    print('✅ 练习 1 通过：新工具能注册并被 agent 按名字分发调用')
finally:
    shutil.rmtree(w, ignore_errors=True)

## ✏️ 练习 2：自己写一版 agent 主循环

实现 `my_run_agent(llm, registry, task, max_steps)`：循环里调 `llm.step(messages, registry.schemas())`；`{'type':'final'}` 则返回 `{'done':True,'steps':i,'answer':text}`；否则 `registry.call` 执行、把 tool_use/tool_result 回填进 messages；到上限返回 `{'done':False,'steps':max_steps,'answer':'(上限)'}`。

In [ ]:
def my_run_agent(llm, registry, task, max_steps=10):
    messages = [{'role':'user','content':task}]
    # TODO: for i in range(max_steps):
    #         a = llm.step(messages, registry.schemas())
    #         if a['type']=='final': return {'done':True,'steps':i,'answer':a['text']}
    #         result = registry.call(a['name'], a['args'])
    #         messages.append assistant tool_use ; messages.append user tool_result
    #       return {'done':False,'steps':max_steps,'answer':'(上限)'}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w = tempfile.mkdtemp(prefix='c31_ex2_')
try:
    open(os.path.join(w,'calc.py'),'w').write('def f():\n    return 0\n')
    r = ToolRegistry(w)
    r.register('read_file', read_file, make_schema('read_file','',{},[]))
    r.register('edit_file', edit_file, make_schema('edit_file','',{},[]))
    llm = MockLLM([
        {'type':'tool','name':'read_file','args':{'path':'calc.py'},'id':'a'},
        {'type':'tool','name':'edit_file','args':{'path':'calc.py','old':'return 0','new':'return 42'},'id':'b'},
        {'type':'final','text':'改好了'}])
    res = my_run_agent(llm, r, '把返回值改成 42')
    assert res['done'] is True and res['steps'] == 2
    assert 'return 42' in open(os.path.join(w,'calc.py')).read()
    print('结果:', res)
    print('✅ 练习 2 通过：自己的主循环能驱动多步工具调用直到 final')
finally:
    shutil.rmtree(w, ignore_errors=True)

## ✏️ 练习 3：SWE-bench 式评测函数（resolved 判定）

实现 `swe_eval(work, run_one)`：记录测试文件指纹 → 调 `run_one(work)`(跑 agent) → 返回 dict：
`tests_pass`(bool)、`tampered`(测试文件是否被改)、`resolved`(测试绿 **且** 未篡改)。复用上面的 `test_files_fingerprint` / `tests_pass`。

In [ ]:
def swe_eval(work, run_one):
    # TODO: before = test_files_fingerprint(work); run_one(work)
    #       after = test_files_fingerprint(work); p = tests_pass(work)
    #       tampered = (before!=after); resolved = p and not tampered
    #       return {'tests_pass':p, 'tampered':tampered, 'resolved':resolved}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 正确修复
w1 = build_repo()
try:
    def fix_impl(work):
        edit_file(work, 'calc.py', 'return a - b', 'return a + b')
    v1 = swe_eval(w1, fix_impl)
    assert v1 == {'tests_pass':True,'tampered':False,'resolved':True}, v1
finally:
    shutil.rmtree(w1, ignore_errors=True)
# 改测试作弊
w2 = build_repo()
try:
    def cheat(work):
        edit_file(work, 'test_calc.py', 'assert add(2, 3) == 5', 'assert add(2, 3) == -1')
    v2 = swe_eval(w2, cheat)
    assert v2['tests_pass'] is True and v2['tampered'] is True and v2['resolved'] is False, v2
finally:
    shutil.rmtree(w2, ignore_errors=True)
print('正确修复:', v1, '\n作弊:', v2)
print('✅ 练习 3 通过：resolved 要求测试绿且未篡改——作弊被正确判为未解决')

## ✏️ 练习 4：给 agent 加「步数预算」与轨迹长度上报

实现 `run_agent_budget(llm, registry, task, max_steps)`：在主循环基础上，返回里额外带 `n_tool_calls`(实际调用工具次数)。用它确认：一个会无限调工具(永不 final)的 LLM 会**恰好在 max_steps 停**、且 `n_tool_calls == max_steps`。

In [ ]:
def run_agent_budget(llm, registry, task, max_steps=10):
    messages = [{'role':'user','content':task}]
    n_tool_calls = 0
    # TODO: for i in range(max_steps):
    #         a = llm.step(...); if final -> return {...,'n_tool_calls':n_tool_calls}
    #         否则 n_tool_calls += 1; registry.call; 回填
    #       return {'done':False,'steps':max_steps,'n_tool_calls':n_tool_calls}
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
class LoopForeverLLM:
    '''永远只调 run_tests、从不 final —— 模拟陷入循环的 agent。'''
    def step(self, messages, schemas):
        return {'type':'tool','name':'run_tests','args':{},'id':'x'}
w = build_repo()
try:
    r = ToolRegistry(w)
    r.register('run_tests', run_tests, make_schema('run_tests','',{},[]))
    res = run_agent_budget(LoopForeverLLM(), r, 'task', max_steps=4)
    print('结果:', res)
    assert res['done'] is False and res['steps'] == 4
    assert res['n_tool_calls'] == 4, '4 步预算用满、每步一次工具调用'
    print('✅ 练习 4 通过：步数预算是安全阀——陷入循环的 agent 会被准时叫停')
finally:
    shutil.rmtree(w, ignore_errors=True)

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def list_files(work):
    out = []
    for root, dirs, files in os.walk(work):
        dirs[:] = [d for d in dirs if d != '__pycache__' and not d.startswith('.')]
        for fn in files:
            if fn.endswith('.py'):
                out.append(os.path.relpath(os.path.join(root, fn), work))
    return '\n'.join(sorted(out))

In [ ]:
# 练习 2 参考答案
def my_run_agent(llm, registry, task, max_steps=10):
    messages = [{'role':'user','content':task}]
    for i in range(max_steps):
        a = llm.step(messages, registry.schemas())
        if a['type'] == 'final':
            return {'done': True, 'steps': i, 'answer': a['text']}
        tid = a.get('id', f'c{i}')
        result = registry.call(a['name'], a['args'])
        messages.append({'role':'assistant','content':[
            {'type':'tool_use','id':tid,'name':a['name'],'input':a['args']}]})
        messages.append({'role':'user','content':[
            {'type':'tool_result','tool_use_id':tid,'content':result}]})
    return {'done': False, 'steps': max_steps, 'answer': '(上限)'}

In [ ]:
# 练习 3 参考答案
def swe_eval(work, run_one):
    before = test_files_fingerprint(work)
    run_one(work)
    after = test_files_fingerprint(work)
    p = tests_pass(work)
    tampered = (before != after)
    return {'tests_pass': p, 'tampered': tampered, 'resolved': p and not tampered}

In [ ]:
# 练习 4 参考答案
def run_agent_budget(llm, registry, task, max_steps=10):
    messages = [{'role':'user','content':task}]
    n_tool_calls = 0
    for i in range(max_steps):
        a = llm.step(messages, registry.schemas())
        if a['type'] == 'final':
            return {'done': True, 'steps': i, 'answer': a['text'], 'n_tool_calls': n_tool_calls}
        n_tool_calls += 1
        tid = a.get('id', f'c{i}')
        result = registry.call(a['name'], a['args'])
        messages.append({'role':'assistant','content':[{'type':'tool_use','id':tid,'name':a['name'],'input':a['args']}]})
        messages.append({'role':'user','content':[{'type':'tool_result','tool_use_id':tid,'content':result}]})
    return {'done': False, 'steps': max_steps, 'answer': '(上限)', 'n_tool_calls': n_tool_calls}

---
## 🧪 真实数据胶囊：agent 端到端解一个 SWE 式任务（多文件 + 评测）

一个更接近真实 SWE-bench 的任务：仓库有**多个文件**，bug 藏在被调用的辅助函数里(`utils.py`)，但失败显现在另一处。agent 要**搜索定位→读→改→测**，最后用 `evaluate` 客观判 `resolved`。全程**真跑 pytest**。

In [ ]:
def build_swe_repo():
    '''多文件玩具仓库：bug 在 utils.discount，被 cart.total 调用，测试失败。'''
    w = tempfile.mkdtemp(prefix='c31_capsule_')
    open(os.path.join(w, 'utils.py'), 'w').write(
        'def discount(price, pct):\n'
        '    # BUG: 折扣算反了，应当是 price * (1 - pct)\n'
        '    return price * pct\n')
    open(os.path.join(w, 'cart.py'), 'w').write(
        'from utils import discount\n\n'
        'def total(price, pct):\n    return round(discount(price, pct), 2)\n')
    open(os.path.join(w, 'test_cart.py'), 'w').write(
        'from cart import total\n\n'
        'def test_total():\n    assert total(100, 0.2) == 80.0   # 8 折后应 80\n')
    return w

# 先确认这个仓库确实是红的，且 bug 在 utils.py
_w = build_swe_repo()
try:
    assert tests_pass(_w) is False
    print('SWE 式仓库:', sorted(os.listdir(_w)))
    print('初始 pytest:', run_tests(_w).splitlines()[-1])
finally:
    shutil.rmtree(_w, ignore_errors=True)
print('✅ 胶囊准备就绪：多文件、bug 藏在辅助函数里，需搜索定位')

**🧪 胶囊练习**：写一个 `swe_agent(work)`：注册 `code_search`/`read_file`/`edit_file`/`run_tests`，用 `MockLLM` 给出合理决策序列（搜 `discount` → 读 `utils.py` → 把 `price * pct` 改成 `price * (1 - pct)` → 跑测试 → final），用 `run_agent` 跑；再用 `evaluate(swe_agent, build_swe_repo)` 断言 `resolved=True`。补全骨架。

In [ ]:
def swe_agent(work):
    r = ToolRegistry(work)
    for nm, fn in [('code_search',code_search),('read_file',read_file),
                   ('edit_file',edit_file),('run_tests',run_tests)]:
        r.register(nm, fn, make_schema(nm, nm, {}, []))
    # TODO: 写 MockLLM 决策脚本（搜 discount → 读 utils.py → 改 'price * pct'->'price * (1 - pct)'
    #        → run_tests → final），然后 run_agent(MockLLM(script), r, '修复 test_cart', verbose=False)
    raise NotImplementedError

# verdict = evaluate(swe_agent, build_swe_repo)
raise NotImplementedError

In [ ]:
# 自测
assert verdict['resolved'] is True, 'agent 应正确修好辅助函数里的 bug'
assert verdict['test_tampered'] is False, '不许改测试'
print('SWE 式评测结果:', verdict)
print('✅✅✅ 胶囊练习通过：agent 自主搜索定位→修复辅助函数→真跑 pytest 全绿，resolved！')

In [ ]:
# 📖 胶囊参考答案
def swe_agent(work):
    r = ToolRegistry(work)
    for nm, fn in [('code_search',code_search),('read_file',read_file),
                   ('edit_file',edit_file),('run_tests',run_tests)]:
        r.register(nm, fn, make_schema(nm, nm, {}, []))
    script = [
        {'type':'tool','name':'code_search','args':{'query':'discount'},'id':'1'},
        {'type':'tool','name':'read_file','args':{'path':'utils.py'},'id':'2'},
        {'type':'tool','name':'edit_file',
         'args':{'path':'utils.py','old':'return price * pct','new':'return price * (1 - pct)'},'id':'3'},
        {'type':'tool','name':'run_tests','args':{},'id':'4'},
        {'type':'final','text':'修复了 discount 的折扣方向，测试通过。'},
    ]
    run_agent(MockLLM(script), r, '修复 test_cart 失败的 bug', verbose=False)
verdict = evaluate(swe_agent, build_swe_repo)
print(verdict)

---
## 🔧 接上真实 Claude：Messages API 适配（无 key 自动回退）

把 MockLLM 换成真实 Claude，只需一个**同接口**的 `ClaudeLLM.step()`，主循环与注册表**一字不改**。下面是完整适配 + `get_llm()` 工厂（**无 key 时不会实例化 ClaudeLLM，仅展示**）。

In [ ]:
class ClaudeLLM:
    '''真实 Claude 适配：用 Messages API 的 tool_use/tool_result/stop_reason。'''
    def __init__(self, client, model='claude-sonnet-4-6', system=''):
        self.client = client; self.model = model; self.system = system
    def step(self, messages, tool_schemas):
        resp = self.client.messages.create(
            model=self.model, max_tokens=2048, system=self.system,
            tools=tool_schemas, messages=messages)
        if resp.stop_reason == 'tool_use':
            for block in resp.content:
                if block.type == 'tool_use':
                    return {'type':'tool','name':block.name,'args':block.input,'id':block.id}
        text = ''.join(b.text for b in resp.content if getattr(b,'type',None)=='text')
        return {'type':'final','text':text}

def get_llm(mock_script=None, model='claude-sonnet-4-6', system=''):
    '''有 key+包 → 真 Claude；否则 → MockLLM。两者同一 step() 接口。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            return ClaudeLLM(anthropic.Anthropic(), model=model, system=system)
        except ImportError:
            pass
    return MockLLM(mock_script or [{'type':'final','text':'(MockLLM 回退)'}])

llm = get_llm(mock_script=[{'type':'final','text':'hi'}], system=SYSTEM_PROMPT)
print('拿到的 LLM:', type(llm).__name__, '（无 key 时为 MockLLM）')
assert hasattr(llm, 'step'), '真假 LLM 必须同接口'
# 用回退的 MockLLM 也能跑同一个 run_agent，证明接口一致
demo = run_agent(llm, reg, 'ping', verbose=False)
assert demo['done'] is True
print('✅ 无 key 回退契约兑现：同一 agent 既能用 MockLLM 跑通，也能一行不改切真实 Claude')

In [ ]:
# 清理
shutil.rmtree(WORK, ignore_errors=True)
print('工作区已清理 ✅')

### 小结 & 课程收官
- **工具注册表**：名字→实现+schema 的表；分发时捕获一切异常转友好错误（agent 鲁棒性的命门）。
- **agent 主循环**：tool_use→执行→回填 tool_result→再问，直到 final 或步数上限；回填格式贴合 Anthropic API。
- **系统提示**：把工程纪律(先理解再改、改完必验、**不准改测试作弊**)注入 LLM。
- **端到端**：注册表+循环+提示 = 不到 200 行、不依赖框架、**真能在玩具仓修 bug 跑绿**的完整 agent。
- **SWE-bench 式评测**：补丁让测试通过 = resolved；**校验未篡改测试**防 reward hacking——评测须不可被操纵。
- **真实 Claude**：同接口的 `ClaudeLLM.step()` + `get_llm()` 工厂，无 key 自动回退，主循环一字不改。

🎓 **恭喜你造出了一个完整的编码 agent！** 你已亲手实现 Claude Code 这类产品的骨架——手(文件)、脚(shell)、眼(搜索)、小脑(edit-test 循环)、加上注册表+主循环+系统提示组装成的自主行为，以及客观的 SWE 式评测。剩下与真实产品的差别是**规模与打磨**(更多工具、更强提示、上下文管理、多 agent)，而非**本质**。把 MockLLM 换成真实 Claude，它就能去解真实仓库的 issue。去造你自己的 agent 吧！